In [3]:
# Celda 1: Importar librerías
import json
import pandas as pd

In [4]:
# Celda 2: Leer el archivo JSON
with open("../data/raw/fintech_events_v4.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Total de eventos: {len(raw_data)}")
print(f"Tipo de dato: {type(raw_data)}")

Total de eventos: 2000
Tipo de dato: <class 'list'>


In [5]:
# Celda 3: Ver la estructura del primer evento
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))

{
  "source": "fintech.app",
  "detailType": "event",
  "detail": {
    "id": "6e3566f3-f4d4-4f02-a9f2-d5fb44c4fbab",
    "event": "TRANSFER_SENT",
    "version": "1.0",
    "eventType": "transfer_sent",
    "transactionType": "transfer_sent",
    "eventEntity": "USER",
    "eventStatus": "SUCCESS",
    "payload": {
      "userId": "user_99",
      "name": "Luisa",
      "age": 49,
      "email": "user291@mail.com",
      "city": "Cartagena",
      "segment": "young_professional",
      "timestamp": "2026-04-21T13:16:41.492486+00:00",
      "accountId": "acc_218",
      "amount": 360088,
      "currency": "COP",
      "merchant": "Rappi",
      "category": "transport",
      "paymentMethod": "debit_card",
      "installments": 1,
      "balanceBefore": 691803,
      "balanceAfter": 378654,
      "location": {
        "city": "Bogotá",
        "country": "Colombia"
      }
    },
    "metadata": {
      "device": "web",
      "os": "ios",
      "ip": "192.168.60.177",
      "channel": "

In [6]:
# Celda 4: Ver cuántos eventos hay por tipo
from collections import Counter
tipos = Counter(r["detail"]["event"] for r in raw_data)
for tipo, cantidad in sorted(tipos.items()):
    print(f"  {tipo}: {cantidad}")

  MONEY_ADDED: 266
  PAYMENT_FAILED: 300
  PAYMENT_MADE: 294
  PURCHASE_MADE: 283
  TRANSFER_SENT: 280
  USER_PROFILE_UPDATED: 284
  USER_REGISTERED: 293


In [7]:
# Celda 5: Ver los valores únicos de campos clave
ciudades = set(r["detail"]["payload"].get("city", "") for r in raw_data)
segmentos = set(r["detail"]["payload"].get("segment", "") for r in raw_data)
print("Ciudades:", ciudades)
print("Segmentos:", segmentos)

Ciudades: {'Medellín', 'Cali', 'Bogotá', 'Cartagena', 'Barranquilla'}
Segmentos: {'family', 'young_professional', 'student', 'premium'}


In [ ]:
###ESTA PRUEBA SE REALIZA DESPUES DE TENER LISTO EL ECOMMERCE.APP Y DE HABER PROCESADO LOS DATOS HASTA LA CAPA BRONZE. 
## SE VERIFICA QUE LOS ARCHIVOS PARQUET SE HAYAN GENERADO CORRECTAMENTE Y QUE CONTENGAN LOS DATOS ESPERADOS.
import glob


archivos = glob.glob("data/bronze/events/**/*.parquet", recursive=True)
print(f"Archivos Parquet generados: {len(archivos)}")

for ruta in archivos:
    df = pd.read_parquet(ruta)
    print(f"\n📄 {ruta}")
    print(f"   Filas: {len(df)} | Columnas: {len(df.columns)}")
    print(f"   Sources: {df['source'].unique()}")   # Debe incluir 'ecommerce.app'
    print(f"   Batch IDs: {df['batch_id'].unique()}")

Archivos Parquet generados: 0


In [ ]:
##ESTA PRUEBA SE REALIZA DESPUES DE TENER LISTO EL ECOMMERCE.APP Y DE HABER PROCESADO LOS DATOS HASTA LA CAPA BRONZE. 
## SE VERIFICA QUE LOS ARCHIVOS PARQUET SE HAYAN GENERADO CORRECTAMENTE Y QUE CONTENGAN LOS DATOS ESPERADOS.

import os

print("🔍 VERIFICACIÓN DE CAPA BRONZE\n")

# 1. ¿Existen archivos Parquet?
archivos = glob.glob("data/bronze/events/**/*.parquet", recursive=True)
print(f"✅ Archivos Parquet: {len(archivos)}")
assert len(archivos) > 0, "❌ No se encontraron archivos Parquet"

# 2. ¿Tienen el formato correcto?
df_total = pd.concat([pd.read_parquet(f) for f in archivos])
print(f"✅ Total de eventos: {len(df_total)}")

# 3. ¿Tienen los metadatos de ingesta?
columnas_requeridas = ["event_id", "source", "event", "user_id",
                        "ingestion_timestamp", "source_file", "batch_id",
                        "ingestion_date", "is_duplicate"]
for col in columnas_requeridas:
    assert col in df_total.columns, f"❌ Falta columna: {col}"
print(f"✅ Columnas requeridas presentes: {len(columnas_requeridas)}")

# 4. ¿Los eventos del ecommerce llegaron?
sources = df_total['source'].unique()
print(f"✅ Fuentes de datos: {sources}")

# 5. ¿Hay duplicados?
dupes = df_total['is_duplicate'].sum()
print(f"{'⚠️' if dupes > 0 else '✅'} Duplicados detectados: {dupes}")

# 6. Distribución por tipo de evento
print(f"\n📊 Eventos por tipo:")
print(df_total['event'].value_counts().to_string())

print("\n🎉 Verificación completada exitosamente")